In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Order_Details.xlsx to Order_Details (2).xlsx
Saving List_of_Orders.xlsx to List_of_Orders (2).xlsx
Saving Sales_target.xlsx to Sales_target (2).xlsx


In [ ]:
Order_Details_df = pd.read_excel("Order_Details.xlsx")
List_of_Orders_df = pd.read_excel("List_of_Orders.xlsx")
Sales_target_df = pd.read_excel("Sales_target.xlsx")

In [ ]:
Order_Details_df.info() # Order_Details
List_of_Orders_df.info()  # List_of_Orders
Sales_target_df.info()   # Sales_target

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Order ID      1500 non-null   object
 1   Amount        1500 non-null   int64 
 2   Profit        1500 non-null   int64 
 3   Quantity      1500 non-null   int64 
 4   Category      1500 non-null   object
 5   Sub-Category  1500 non-null   object
dtypes: int64(3), object(3)
memory usage: 70.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Order ID      500 non-null    object
 1   Order Date    500 non-null    object
 2   CustomerName  500 non-null    object
 3   State         500 non-null    object
 4   City          500 non-null    object
dtypes: object(5)
memory usage: 19.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 3

In [ ]:
Order_Details_df.isnull().sum()


,0
Order ID,0
Amount,0
Profit,0
Quantity,0
Category,0
Sub-Category,0


In [ ]:
Order_Details_df.shape


(1500, 6)

In [ ]:
List_of_Orders_df.shape

(500, 5)

In [ ]:
Sales_target_df.shape

(36, 3)

In [ ]:
List_of_Orders_df.isnull().sum()

,0
Order ID,0
Order Date,0
CustomerName,0
State,0
City,0


In [ ]:
Sales_target_df.isnull().sum()

,0
Month of Order Date,0
Category,0
Target,0


In [ ]:
# Part 1: Sales and Profitability Analysis

#Q1. Merge the List of Orders and Order Details datasets on the basis of OrderID. Calculate the total sales (Amount) for each category across all orders.

merged_df = pd.merge(List_of_Orders_df,Order_Details_df, on="Order ID" , how = "inner")
category_sales = merged_df.groupby("Category")["Amount"].sum()


In [ ]:
category_sales

,Amount
Category,
Clothing,139054
Electronics,165267
Furniture,127181


In [ ]:
# For each category, calculate the average profit per order and total profit margin (profit as a percentage of Amount)

# So to find the Average Profit per Order , we will find the Total Profit and then divide it with number of unique orders.

category_orders = (
    merged_df.groupby("Category", as_index=False)["Order ID"]
    .nunique()
)

# Finding the profit for each category
category_profit = (
    merged_df.groupby("Category", as_index=False)["Profit"]
    .sum()
)

In [ ]:
print(category_orders)
print(category_profit)

      Category  Order ID
0     Clothing       393
1  Electronics       204
2    Furniture       186
      Category  Profit
0     Clothing   11163
1  Electronics   10494
2    Furniture    2298


In [ ]:
# Profit for each category is : avg_profit_per_category

# Profit : Total Profit
# OrderID : Number of Unique Order

avg_profit_per_category = category_profit["Profit"] / category_orders["Order ID"]
category_profit["Average Profit per Order"] = avg_profit_per_category



In [ ]:
print(category_profit[["Category", "Average Profit per Order"]])

      Category  Average Profit per Order
0     Clothing                 28.404580
1  Electronics                 51.441176
2    Furniture                 12.354839


In [ ]:
# Total Profit Margin for each category : profit for that category / total sales for that category

profit_margin_df = (
    merged_df.groupby("Category", as_index=False)[["Amount", "Profit"]]
    .sum()
)

profit_margin_df["Profit Margin (%)"] = (
    profit_margin_df["Profit"] /
    profit_margin_df["Amount"]
) * 100

print(profit_margin_df)

      Category  Amount  Profit  Profit Margin (%)
0     Clothing  139054   11163           8.027817
1  Electronics  165267   10494           6.349725
2    Furniture  127181    2298           1.806874


In [ ]:
# Identify the top-performing and underperforming categories based on these metrics. Also, suggest reasons for their performance differences.

# Top performing product is Clothing with Amount(Rs 139054), profit(Rs 11163) , average profit of (Rs 28.4 ) per order and a profit margin of (8%).
# Underperforming product is the  Furniture with Amount(Rs 127181) , profit(Rs 2298) only, average profit of (Rs 12.35) per order and profit margin of (1.8%) only.

# SUGGESTIONS :
# 1.  Improve the pricing and discount strategy for the Furniture Category while optimizing operational cost and procurement to increase the profit margin
# 2.  Improve marketing and promotional efforts for the Furniture Category , by targeting the right customer segments to increase the demand and improve sales and performance.

In [ ]:
# Using the Sales Target dataset, calculate the percentage change in target sales for the Furniture category month-over-month.

# 1. First Filter Furniture from the Category.
# 2. Then calculate the percentage change over month to month .

furniture_target = Sales_target_df[Sales_target_df["Category"] == "Furniture"].copy()
furniture_target["Target Change (%)"] = (
    furniture_target["Target"].pct_change() * 100
)

In [ ]:
print(furniture_target)

   Month of Order Date   Category  Target  Target Change (%)
0           2026-04-18  Furniture   10400                NaN
1           2026-05-18  Furniture   10500           0.961538
2           2026-06-18  Furniture   10600           0.952381
3           2026-07-18  Furniture   10800           1.886792
4           2026-08-18  Furniture   10900           0.925926
5           2026-09-18  Furniture   11000           0.917431
6           2026-10-18  Furniture   11100           0.909091
7           2026-11-18  Furniture   11300           1.801802
8           2026-12-18  Furniture   11400           0.884956
9           2026-01-19  Furniture   11500           0.877193
10          2026-02-19  Furniture   11600           0.869565
11          2026-03-19  Furniture   11800           1.724138


In [ ]:
#From the List of Orders dataset, identify the top 5 states with the highest order count. For each of these states, calculate the total sales and average profit.

state_summary = merged_df.groupby("State", as_index=False).agg({
    "Order ID": "nunique",
    "Amount": "sum",
    "Profit": "mean"
})

state_summary.rename(columns={
    "Order ID": "Order Count",
    "Amount": "Total Sales",
    "Profit": "Average Profit"
}, inplace=True)

top_states = state_summary.sort_values(
    by="Order Count",
    ascending=False
).head(5)

print(top_states)

             State  Order Count  Total Sales  Average Profit
10  Madhya Pradesh          101       105140       16.326471
11     Maharashtra           90        95348       21.296552
14       Rajasthan           32        21149       16.986486
4          Gujarat           27        21058        5.344828
13          Punjab           25        16786      -10.150000
